# MultiMedAI — High-quality medical illustration generation (FREE Colab GPU)

Your laptop is CPU-only, so it can only run tiny SD-Turbo (cartoonish). This notebook uses a **free Colab T4 GPU** to run **SDXL** (and optionally **FLUX.1-schnell**) for far more realistic / illustrative images.

**Honesty:** these are still **synthetic** — general models, not trained on real anatomy, so treat outputs as *illustrations*, not accurate medical references. And diffusion still can't render clean text labels (use the app's Gemini auto-label overlay for that).

### Run: Colab → Upload → Runtime→T4 GPU → Run all.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
!pip -q install diffusers==0.31.0 transformers==4.46.0 accelerate==1.6.0

## Option A — SDXL (fits a free T4; strong, reliable)

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16,
    variant='fp16', use_safetensors=True).to('cuda')

prompt = 'a detailed anatomical medical illustration of the human brain, textbook style, clean white background, realistic proportions, high detail, sharp, labeled diagram style'
negative = 'cartoon, childish, distorted, deformed, extra parts, tiled, dotted background, watermark, blurry, low quality'
img = pipe(prompt, negative_prompt=negative, num_inference_steps=35,
           guidance_scale=7.0, height=1024, width=1024).images[0]
img.save('sdxl_out.png'); img

## Option B — FLUX.1-schnell (best realism/illustration, Apache-2.0)
Heavier (~needs high-RAM GPU; may be slow/OOM on a base T4). Skip if it errors.

In [ ]:
try:
    from diffusers import FluxPipeline
    flux = FluxPipeline.from_pretrained('black-forest-labs/FLUX.1-schnell',
                                        torch_dtype=torch.bfloat16)
    flux.enable_model_cpu_offload()   # helps fit on smaller GPUs
    img2 = flux(prompt, num_inference_steps=4, guidance_scale=0.0,
                height=1024, width=1024).images[0]
    img2.save('flux_out.png'); img2
except Exception as e:
    print('FLUX not available on this GPU:', str(e)[:120])

## Download your images

In [ ]:
try:
    from google.colab import files
    import os
    for f in ['sdxl_out.png', 'flux_out.png']:
        if os.path.exists(f): files.download(f)
except Exception:
    print('Download sdxl_out.png / flux_out.png manually.')

## Notes
- **SDXL** at 1024px, 35 steps, guidance ~7 gives clean, illustrative results (vs cartoonish SD-Turbo on CPU).
- For **labels**, generate here, bring the image back, upload it in the app, and ask *“label this”* — the app overlays real Gemini-detected labels.
- Still **synthetic** — not a substitute for real scans (use the app's 80k retrieval bank for real images).